<a href="https://colab.research.google.com/github/harsh3-web/Ecommerce-Recommender/blob/main/notebooks/07_multimodal_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 07 · Multimodal model training

PyTorch model: user embedding · product vector, where the product vector = ID embedding + projected ResNet50 + projected MiniLM features, mixed with neighbours by a trainable **GCN** layer (residual). Trained with **BPR loss** and negative sampling, early stopping on validation NDCG, plus an **ablation** without the GCN. Saves `multimodal_gcn.pt`.

In [45]:
!git clone https://github.com/harsh3-web/Ecommerce-Recommender.git
%cd Ecommerce-Recommender
# torch_geometric provides GCNConv
!pip install -q sentence-transformers torch_geometric

In [46]:
from google.colab import drive
drive.mount("/content/drive")
!cp /content/drive/MyDrive/Ecommerce-Recommender-data/*.csv .
!ls *.csv

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
browsing_history_expanded.csv  products_expanded.csv
ground_truth_tastes.csv        purchases_expanded.csv
product_images_expanded.csv    users_expanded.csv


In [47]:
import numpy as np
import pandas as pd

products  = pd.read_csv("products_expanded.csv")
users     = pd.read_csv("users_expanded.csv")
# parse_dates -> timestamps become real dates, so we can find each user's LAST purchase
purchases = pd.read_csv("purchases_expanded.csv", parse_dates=["timestamp"])
browsing  = pd.read_csv("browsing_history_expanded.csv", parse_dates=["timestamp"])
n_users, n_products = len(users), len(products)

In [48]:
# sort by time, then take the LAST row per user = their most recent purchase
last = purchases.sort_values("timestamp").groupby("user_id").tail(1)

n_buys = purchases.groupby("user_id").size()          # purchases per user
eligible = n_buys[n_buys >= 2].index                  # need >= 1 purchase LEFT for training
test_items = last[last["user_id"].isin(eligible)].set_index("user_id")["product_id"]
print(len(test_items), "users will be evaluated")     # test_items[u] = the hidden product for user u

455 users will be evaluated


In [49]:
def drop_test(df):
    """Remove every row where a user interacts with THEIR OWN hidden test product."""
    hidden = df["user_id"].map(test_items)     # hidden product for each row's user (NaN if not evaluated)
    return df[df["product_id"] != hidden].copy()

train_purchases = drop_test(purchases)
train_browsing  = drop_test(browsing)
print(f"Purchases: {len(purchases)} -> {len(train_purchases)} | Browsing: {len(browsing)} -> {len(train_browsing)}")

Purchases: 1901 -> 1446 | Browsing: 6269 -> 5814


In [50]:
rng = np.random.default_rng(42)
eval_users = test_items.index.to_numpy()
rng.shuffle(eval_users)                       # random order (fixed by the seed)
half = len(eval_users) // 2
val_users, test_users = eval_users[:half], eval_users[half:]
print(len(val_users), "validation users |", len(test_users), "test users")

227 validation users | 228 test users


In [51]:
def hit_at_k(ranked_ids, true_id, k=10):
    """1 if the hidden product appears anywhere in the top k, else 0."""
    return float(true_id in ranked_ids[:k])

def ndcg_at_k(ranked_ids, true_id, k=10):
    """Like hit rate, but rewards ranking the hidden product HIGHER."""
    top = list(ranked_ids[:k])
    if true_id not in top:
        return 0.0
    rank = top.index(true_id) + 1             # position 1, 2, ... 10
    return 1 / np.log2(rank + 1)              # rank 1 -> 1.0, rank 2 -> 0.63, rank 10 -> 0.29

In [52]:
def evaluate(recommend_fn, user_ids, k=10):
    """Average HR@k and NDCG@k over users. recommend_fn(u) must return a table sorted best-first."""
    hits, ndcgs = [], []
    for u in user_ids:
        ranked = recommend_fn(u)["product_id"].tolist()
        hits.append(hit_at_k(ranked, test_items[u], k))
        ndcgs.append(ndcg_at_k(ranked, test_items[u], k))
    return {"HR@10": np.mean(hits), "NDCG@10": np.mean(ndcgs)}

In [53]:
from scipy.sparse import csr_matrix
BROWSE_W, BUY_W = 1.0, 4.0
b = train_browsing[["user_id", "product_id"]].assign(weight=BROWSE_W)
p = train_purchases[["user_id", "product_id"]].assign(weight=BUY_W)
inter = pd.concat([b, p]).groupby(["user_id", "product_id"], as_index=False)["weight"].sum()

R = csr_matrix((inter["weight"].values, (inter["user_id"].values - 1, inter["product_id"].values - 1)),
               shape=(n_users, n_products))

In [54]:
from sklearn.metrics.pairwise import cosine_similarity
item_sim = cosine_similarity(R.T)        # 200 x 200, built from TRAINING data only
np.fill_diagonal(item_sim, 0)

# Dictionary: user_id -> set of products they already touched (in training). Fast to look up.
seen = inter.groupby("user_id")["product_id"].apply(set).to_dict()

In [55]:
def to_recs(scores, uid, source):
    """Array of 200 scores (product_id order) -> ranked table without already-seen products."""
    recs = products.copy()
    recs["score"], recs["source"] = scores, source
    recs = recs[~recs["product_id"].isin(seen.get(uid, set()))]   # .get -> empty set if user unknown
    return recs.sort_values("score", ascending=False)

In [56]:
def random_recs(uid):
    return to_recs(rng.random(n_products), uid, "Random")       # random scores -> random ranking

# How often each product was bought in TRAINING; reindex fills never-bought products with 0
pop = train_purchases["product_id"].value_counts().reindex(products["product_id"], fill_value=0).values
def popular(uid):
    return to_recs(pop.astype(float), uid, "Popular")            # same bestseller list for everyone

In [57]:
def item_cf(uid):
    user_vec = R[uid - 1].toarray().ravel()      # this user's training weights
    return to_recs(user_vec @ item_sim, uid, "Item-CF")

In [58]:
from sentence_transformers import SentenceTransformer
encoder = SentenceTransformer("all-MiniLM-L6-v2")
texts = (products["category"] + ". " + products["product_name"] + ". " + products["description"]).tolist()
item_emb = encoder.encode(texts, normalize_embeddings=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [59]:
def content_emb(uid):
    mine = inter[inter["user_id"] == uid]                          # TRAINING interactions only
    w = mine["weight"].values[:, None]
    profile = (w * item_emb[mine["product_id"].values - 1]).sum(axis=0) / w.sum()
    profile = profile / np.linalg.norm(profile)
    return to_recs(item_emb @ profile, uid, "Content-Emb")

In [60]:
def minmax(s):
    spread = s.max() - s.min()
    return (s - s.min()) / spread if spread > 0 else s * 0

def hybrid_v2(uid, alpha=0.5):
    cf = minmax(item_cf(uid).set_index("product_id")["score"])
    cb = minmax(content_emb(uid).set_index("product_id")["score"])
    combined = (alpha * cf + (1 - alpha) * cb).sort_values(ascending=False)
    return products.set_index("product_id").loc[combined.index].reset_index().assign(score=combined.values)

In [61]:
##Load the image embeddings

DATA = "/content/drive/MyDrive/Ecommerce-Recommender-data"
!cp "{DATA}/image_emb.npy" .
image_emb = np.load("image_emb.npy")      # (200, 2048), saved on Day 4, rows in product_id order
print(image_emb.shape, "| text:", item_emb.shape)   # item_emb = text embeddings from the copied cells

(200, 2048) | text: (200, 384)


In [62]:
##The user × product purchase matrix (training data only)
from scipy.sparse import csr_matrix

# B[u, p] = 1 if user u bought product p (TRAINING purchases only -> no leakage of test items)
B = csr_matrix((np.ones(len(train_purchases)),
                (train_purchases["user_id"].values - 1, train_purchases["product_id"].values - 1)),
               shape=(n_users, n_products))
B.data[:] = 1.0                       # make sure every entry is exactly 1, even if repeated
print(B.shape, "| purchases:", B.nnz)

(500, 200) | purchases: 1446


In [63]:
##Co-purchase counts and the adjacency matrix
# B.T @ B -> C[i, j] = number of users who bought BOTH product i and product j
C = (B.T @ B).toarray()
np.fill_diagonal(C, 0)                # a product is not its own neighbour

A = (C > 0).astype(float)             # adjacency matrix: 1 if co-purchased at least once, else 0
print("Edges:", int(A.sum() / 2))     # /2 because A is symmetric (i-j and j-i are the same edge)

Edges: 1753


In [64]:
##Graph statistics
degree = A.sum(axis=1)                          # number of neighbours of each product
print(f"Average neighbours per product: {degree.mean():.1f}")
print(f"Max neighbours: {int(degree.max())} | Min: {int(degree.min())}")
print(f"Isolated products (no edges): {(degree == 0).sum()}")    # cold-start in graph terms
print(f"Graph density: {A.sum() / (n_products * (n_products - 1)):.1%}")  # share of possible edges present

Average neighbours per product: 17.5
Max neighbours: 79 | Min: 0
Isolated products (no edges): 7
Graph density: 8.8%


In [65]:
##Convert to PyTorch Geometric's format
import torch

src, dst = np.nonzero(A)              # row and column positions of every 1 in A
edge_index = torch.tensor(np.vstack([src, dst]), dtype=torch.long)
print(edge_index.shape)               # (2, number_of_directed_edges)
print(edge_index[:, :5])              # first 5 edges: top row = from, bottom row = to

torch.Size([2, 3506])
tensor([[0, 0, 0, 0, 0],
        [1, 2, 5, 7, 9]])


In [72]:
##: Turn inputs into PyTorch tensors
import torch
import copy

img_t  = torch.tensor(image_emb, dtype=torch.float32)   # (200, 2048) ResNet features from Day 4
txt_t  = torch.tensor(item_emb,  dtype=torch.float32)   # (200, 384)  MiniLM features
edge_t = edge_index                                     # (2, edges)  co-purchase graph from Day 5
print(img_t.shape, txt_t.shape, edge_t.shape)

torch.Size([200, 2048]) torch.Size([200, 384]) torch.Size([2, 3506])


In [73]:
#Training examples (positives)
# Every training browse AND purchase is a "positive" (user liked this product)
# A product that was browsed AND bought appears twice -> the model sees purchases more often
pos = pd.concat([train_browsing[["user_id", "product_id"]],
                 train_purchases[["user_id", "product_id"]]])
pos_users = pos["user_id"].values - 1        # 0-based indices for PyTorch embeddings
pos_items = pos["product_id"].values - 1
print("Training pairs:", len(pos_users))

Training pairs: 7260


In [74]:
#Negative sampling
def sample_negatives(users0):
    """For each user, pick a random product they have NOT interacted with."""
    neg = rng.integers(0, n_products, len(users0))              # random product indices 0..199
    for k in range(len(neg)):
        while (neg[k] + 1) in seen.get(users0[k] + 1, ()):      # +1: 'seen' uses 1-based ids
            neg[k] = rng.integers(0, n_products)                # re-draw until it's truly unseen
    return neg

In [75]:
##The model
import torch.nn as nn
from torch_geometric.nn import GCNConv

class MultiModalRec(nn.Module):
    def __init__(self, n_users, n_products, dim=64, use_gcn=True):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, dim)        # one learnable 64-number vector per user
        self.item_emb = nn.Embedding(n_products, dim)     # one learnable vector per product (its "ID" part)
        nn.init.normal_(self.user_emb.weight, std=0.1)    # start with small random values
        nn.init.normal_(self.item_emb.weight, std=0.1)
        self.img_proj = nn.Linear(2048, dim)              # compress image features 2048 -> 64
        self.txt_proj = nn.Linear(384, dim)               # compress text features 384 -> 64
        self.gcn = GCNConv(dim, dim) if use_gcn else None # trainable GCN layer (W is learned now!)

    def product_vectors(self, img, txt, edge_index):
        x = self.item_emb.weight + self.img_proj(img) + self.txt_proj(txt)   # fuse 3 sources by adding
        if self.gcn is not None:
            x = x + self.gcn(x, edge_index)               # residual: own vector + neighbour info
        return x                                          # (200, 64)

    def forward(self, users, items, prod_vecs):
        return (self.user_emb(users) * prod_vecs[items]).sum(dim=1)   # dot product = score

In [76]:
#The BPR loss
def bpr_loss(pos_scores, neg_scores):
    """Bayesian Personalized Ranking: reward pos_score > neg_score."""
    return -torch.nn.functional.logsigmoid(pos_scores - neg_scores).mean()

In [77]:
##Score every user-product pair, and plug into our evaluation
@torch.no_grad()                                   # no gradients needed for scoring
def all_scores(model):
    model.eval()
    prod = model.product_vectors(img_t, txt_t, edge_t)
    return (model.user_emb.weight @ prod.T).numpy()   # (500, 200): score for every user x product

def make_model_recommender(S, name):
    return lambda uid: to_recs(S[uid - 1], uid, name)   # reuse to_recs -> same evaluation as before

In [78]:
#One epoch of training
def train_epoch(model, opt, batch_size=1024):
    model.train()
    order = rng.permutation(len(pos_users))                    # shuffle training pairs every epoch
    total = 0.0
    for start in range(0, len(order), batch_size):             # go through data in mini-batches
        idx = order[start:start + batch_size]
        u, i = torch.tensor(pos_users[idx]), torch.tensor(pos_items[idx])
        j = torch.tensor(sample_negatives(pos_users[idx]))     # fresh negatives each time
        prod = model.product_vectors(img_t, txt_t, edge_t)     # recompute: weights changed last step
        loss = bpr_loss(model(u, i, prod), model(u, j, prod))
        opt.zero_grad(); loss.backward(); opt.step()           # the standard 3-step PyTorch update
        total += loss.item() * len(idx)
    return total / len(order)                                  # average loss this epoch

In [79]:
##Full training with early stopping
def fit(use_gcn, epochs=60, check_every=5):
    torch.manual_seed(42)                                       # same starting weights every run
    model = MultiModalRec(n_users, n_products, use_gcn=use_gcn)
    opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-5)   # weight_decay = L2 regularisation
    best_ndcg, best_state = -1, None
    for ep in range(1, epochs + 1):
        loss = train_epoch(model, opt)
        if ep % check_every == 0:                               # every 5 epochs, check VALIDATION users
            ndcg = evaluate(make_model_recommender(all_scores(model), "MM"), val_users)["NDCG@10"]
            print(f"epoch {ep:3d} | loss {loss:.4f} | val NDCG {ndcg:.3f}")
            if ndcg > best_ndcg:
                best_ndcg, best_state = ndcg, copy.deepcopy(model.state_dict())   # remember best weights
    model.load_state_dict(best_state)                           # go back to the best checkpoint
    return model

In [80]:
##Train the full model (with GCN)
model_gcn = fit(use_gcn=True)

epoch   5 | loss 0.2096 | val NDCG 0.154
epoch  10 | loss 0.1404 | val NDCG 0.170
epoch  15 | loss 0.1040 | val NDCG 0.145
epoch  20 | loss 0.0841 | val NDCG 0.152
epoch  25 | loss 0.0608 | val NDCG 0.145
epoch  30 | loss 0.0481 | val NDCG 0.161
epoch  35 | loss 0.0401 | val NDCG 0.158
epoch  40 | loss 0.0346 | val NDCG 0.165
epoch  45 | loss 0.0292 | val NDCG 0.144
epoch  50 | loss 0.0234 | val NDCG 0.137
epoch  55 | loss 0.0221 | val NDCG 0.134
epoch  60 | loss 0.0169 | val NDCG 0.128


In [81]:
#Ablation: the same model WITHOUT the GCN
model_nogcn = fit(use_gcn=False)   # identical setup, only the GCN removed

epoch   5 | loss 0.2296 | val NDCG 0.167
epoch  10 | loss 0.1318 | val NDCG 0.183
epoch  15 | loss 0.0945 | val NDCG 0.178
epoch  20 | loss 0.0696 | val NDCG 0.158
epoch  25 | loss 0.0507 | val NDCG 0.157
epoch  30 | loss 0.0422 | val NDCG 0.156
epoch  35 | loss 0.0319 | val NDCG 0.129
epoch  40 | loss 0.0265 | val NDCG 0.127
epoch  45 | loss 0.0226 | val NDCG 0.136
epoch  50 | loss 0.0196 | val NDCG 0.136
epoch  55 | loss 0.0171 | val NDCG 0.124
epoch  60 | loss 0.0172 | val NDCG 0.129


In [82]:
##Final comparison on TEST users
untrained = MultiModalRec(n_users, n_products)      # random weights = what the ORIGINAL app did

final = {
    "Popular": popular, "Item-CF": item_cf,
    "Hybrid Day 3 (a=0.75)": lambda u: hybrid_v2(u, alpha=0.75),
    "Untrained model (original)": make_model_recommender(all_scores(untrained), "Untrained"),
    "Trained, no GCN": make_model_recommender(all_scores(model_nogcn), "NoGCN"),
    "Trained, with GCN": make_model_recommender(all_scores(model_gcn), "GCN"),
}
pd.DataFrame([{"Method": n, **evaluate(f, test_users)} for n, f in final.items()]).set_index("Method").round(3)

,HR@10,NDCG@10
Method,,
Popular,0.211,0.108
Item-CF,0.373,0.196
Hybrid Day 3 (a=0.75),0.364,0.197
Untrained model (original),0.061,0.027
"Trained, no GCN",0.325,0.156
"Trained, with GCN",0.294,0.170


In [83]:
###Save the trained model
torch.save(model_gcn.state_dict(), "multimodal_gcn.pt")   # state_dict = all learned weights
!cp multimodal_gcn.pt "{DATA}/"
print("Saved model weights")

Saved model weights


## Day 6 results - trained multimodal model (BPR, early stopping)
Model: user emb + [product ID emb + Linear(ResNet 2048->64) + Linear(MiniLM 384->64)]
       -> GCNConv with residual -> dot-product score. ~200k parameters, ~7k training pairs.
Training: BPR loss, random negative sampling, Adam lr=0.01, weight_decay=1e-5,
          batch 1024, early stopping on validation NDCG (best at epoch 10 for both runs)

| Method                     | HR@10 | NDCG@10 |
|----------------------------|-------|---------|
| Untrained (original repo)  | 0.061 | 0.027   |  <- = random: original app was noise
| Trained, no GCN            | 0.325 | 0.156   |
| Trained, with GCN          | 0.294 | 0.170   |
| Item-CF                    | 0.373 | 0.196   |
| Hybrid Day 3 (a=0.75)      | 0.364 | 0.197   |  <- final model

Takeaways:
- Training: ~5x better than untrained; beats Popular
- Overfits after ~10 epochs (train loss falls, val NDCG drops) -> early stopping
- Below item-CF: too many parameters for too little data; simple methods win on small data
- Ablation: GCN gives no significant difference (mixed HR/NDCG, within noise)
- Next: smaller dim, more regularisation, more data